# 🚀 Phase 2: Final Test Evaluation & Comparative Analysis
**Paper Title:** Early-Stopping Activation Steering for Hallucination Mitigation in Vietnamese Domain-Specific RAG
**Dataset:** 14,700 Specialized Medical QA (`vietnamese_medical_halueval_15k_specialized.json`)
**Model:** `Qwen/Qwen2.5-7B-Instruct` (4-bit NF4 Quantization)

---

In [ ]:
# Cell 1: Environment Setup & Dependencies
!pip install -q bitsandbytes accelerate transformers torch scikit-learn rouge-score tqdm
print('✅ Dependencies successfully installed!')

In [ ]:
# Cell 2: Data Loading, Split Reproducibility & Load Phase 1 Checkpoint Artifacts
import os, json, glob, random, time
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_FILENAME = 'vietnamese_medical_halueval_15k_specialized.json'
search_paths = [
    f'/kaggle/input/**/{DATA_FILENAME}',
    f'/kaggle/input/{DATA_FILENAME}',
    f'data/{DATA_FILENAME}',
    f'./{DATA_FILENAME}',
    f'E:/Paper_Steering_VN_15K/data/{DATA_FILENAME}'
]

data_path = None
for pattern in search_paths:
    matches = glob.glob(pattern, recursive=True)
    if matches:
        data_path = matches[0]
        break

with open(data_path, 'r', encoding='utf-8') as f:
    raw_dataset = json.load(f)

shuffled_records = list(raw_dataset)
random.shuffle(shuffled_records)

n_total = len(shuffled_records)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)
test_records = shuffled_records[n_train + n_val:]
print(f'📌 Test Split Loaded: {len(test_records):,} records (Question-Disjoint)')

# --- LOAD CHECKPOINTS TỪ PHASE 1 ---
config_paths = glob.glob('/kaggle/input/**/steering_config.json', recursive=True) + ['./steering_config.json']
v_steer_paths = glob.glob('/kaggle/input/**/v_steer.pt', recursive=True) + ['./v_steer.pt']
v_rand_paths = glob.glob('/kaggle/input/**/v_rand.pt', recursive=True) + ['./v_rand.pt']

if not config_paths or not os.path.exists(config_paths[0]):
    raise FileNotFoundError('❌ Could not locate steering_config.json. Please upload Phase 1 Output as Kaggle Input.')

with open(config_paths[0], 'r', encoding='utf-8') as f:
    steering_config = json.load(f)

best_layer = steering_config['best_layer']
best_alpha = steering_config['best_alpha']
best_K = steering_config['best_K']

v_steer = torch.load(v_steer_paths[0]).to(dtype=torch.bfloat16, device='cuda')
v_rand = torch.load(v_rand_paths[0]).to(dtype=torch.bfloat16, device='cuda')

print(f'✅ Phase 1 Artifacts Loaded:')
print(f'   - Optimal Layer Index: {best_layer}')
print(f'   - Optimal Alpha Scaling: {best_alpha}')
print(f'   - Optimal Early-Stop Step K: {best_K}')
print(f'   - Steering Tensor Shape: {v_steer.shape}')

In [ ]:
# Cell 3: Initialize Qwen2.5-7B Engine
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'

print(f'⌛ Loading model {MODEL_NAME} in 4-bit NF4...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
model.eval()
print('✅ Qwen2.5-7B-Instruct successfully loaded on GPU!')

In [ ]:
# Cell 4: Final 5-Way Comparative Test Evaluation
from rouge_score import rouge_scorer
from tqdm import tqdm

PROMPT_TEMPLATE = """Dựa vào ngữ cảnh y học sau đây, hãy trả lời câu hỏi:
Ngữ cảnh: {context}
Câu hỏi: {question}
Trả lời: """

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

class EarlyStoppingSteeringHook:
    def __init__(self, layer_idx, v_vector, alpha=20.0, K=8):
        self.layer_idx = layer_idx
        self.v_vector = v_vector
        self.alpha = alpha
        self.K = K
        self.step_counter = 0
        self.handle = None
        
    def hook_fn(self, module, inputs, output):
        if self.step_counter < self.K:
            if isinstance(output, tuple):
                h = output[0]
                v = self.v_vector.to(h.device)
                h[:, -1, :] = h[:, -1, :] + self.alpha * v
                output = (h,) + output[1:]
            else:
                v = self.v_vector.to(output.device)
                output[:, -1, :] = output[:, -1, :] + self.alpha * v
        self.step_counter += 1
        return output
        
    def register(self, model):
        layer_module = model.model.layers[self.layer_idx]
        self.step_counter = 0
        self.handle = layer_module.register_forward_hook(self.hook_fn)
        
    def remove(self):
        if self.handle:
            self.handle.remove()
            self.handle = None

print('\n🚀 RUNNING FINAL 5-WAY COMPARATIVE TEST EVALUATION ON TEST SET...')

TEST_EVAL_LIMIT = min(500, len(test_records))
eval_test_records = test_records[:TEST_EVAL_LIMIT]

methods = [
    ('Vanilla Qwen2.5-7B', None, 0.0, 0),
    ('Full Steering (K=∞)', v_steer, best_alpha, 999),
    (f'Early-Stop (K={best_K})', v_steer, best_alpha, best_K),
    ('Ctrl: Random Direction', v_rand, best_alpha, best_K),
    ('Ctrl: Sign-Flipped', -v_steer, best_alpha, best_K)
]

eval_results = {}

for name, vector, alpha, K in methods:
    print(f'\n⌛ Evaluating Method: {name} ({TEST_EVAL_LIMIT} questions)...')
    rouges, lengths, latencies = [], [], []
    
    for rec in tqdm(eval_test_records, desc=f'Testing {name}'):
        ctx = rec.get('knowledge_context', rec.get('context', ''))
        q = rec['question']
        ref = rec['right_answer']
        prompt = PROMPT_TEMPLATE.format(context=ctx, question=q)
        inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
        
        t0 = time.time()
        if vector is not None:
            hook = EarlyStoppingSteeringHook(best_layer, vector, alpha=alpha, K=K)
            hook.register(model)
            with torch.no_grad():
                torch.manual_seed(42)
                out_ids = model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.1, top_p=0.85)
            hook.remove()
        else:
            with torch.no_grad():
                torch.manual_seed(42)
                out_ids = model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.1, top_p=0.85)
        t1 = time.time()
        
        gen_tokens = out_ids[0][inputs.input_ids.shape[1]:]
        gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True)
        
        r_score = scorer.score(ref, gen_text)['rougeL'].fmeasure * 100
        rouges.append(r_score)
        lengths.append(len(gen_tokens))
        latencies.append((t1 - t0) * 1000)
    
    mean_r = np.mean(rouges)
    mean_len = np.mean(lengths)
    mean_lat = np.mean(latencies)
    tput = mean_len / (mean_lat / 1000) if mean_lat > 0 else 0.0
    
    eval_results[name] = {
        'rougeL': mean_r,
        'mean_len': mean_len,
        'latency_ms': mean_lat,
        'throughput': tput
    }
    print(f"Result [{name}]: ROUGE-L={mean_r:.2f}% | Len={mean_len:.2f} tok | Latency={mean_lat:.2f} ms | Throughput={tput:.2f} tok/s")

In [ ]:
# Cell 5: Formatted Summary Report & Exporting Results JSON
output_json_path = 'steering_vn_15k_results.json'
with open(output_json_path, 'w', encoding='utf-8') as f:
    json.dump(eval_results, f, indent=2, ensure_ascii=False)

print('\n=========================================================================================')
print('📊 FINAL EXPERIMENTAL SUMMARY TABLE (Vietnamese Medical 15K Dataset - Qwen2.5-7B-Instruct)')
print('=========================================================================================')
print(f"{'Method':<28} | {'ROUGE-L (%)':<12} | {'Len (tok)':<10} | {'Latency (ms)':<14} | {'Throughput (tok/s)':<18}")
print('-'*90)
for name, res in eval_results.items():
    print(f"{name:<28} | {res['rougeL']:<12.2f} | {res['mean_len']:<10.2f} | {res['latency_ms']:<14.2f} | {res['throughput']:<18.2f}")
print('=========================================================================================')
print(f'✅ Results saved to {output_json_path}')